# Spectral analysis with the Fourier Transform

This tutorial demonstrates how to use the Fourier Transform methods in **earthkit-transforms** to analyse a timeseries in the frequency domain. We use an hourly 2m temperature dataset for two locations (Reading and Vancouver) to show that every step works when the data hold more than one time-series. We will:

1. compute the forward FFT of an hourly timeseries with `temporal.fft`,
2. inspect the power spectrum and identify the dominant (diurnal) cycle,
3. centre the spectrum with `fftshift`,
4. apply a simple low-pass filter and reconstruct a smoothed signal with `temporal.ifft`, and
5. verify that the forward and inverse transforms round-trip.

The transforms use the `fft` extension of the array namespace of the input data (the Python array API standard), so they run on the native backend of the data and return `xarray` objects.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from earthkit import data as ekd
from earthkit import transforms as ekt

# Hourly 2m temperature timeseries (72 hourly steps over 3 days) for two locations
ds = ekd.from_source("sample", "era5-timeseries-multiple.nc").to_xarray()
da = ds["t2m"]

da.plot.line(x="valid_time", hue="location", marker="o")
plt.title("Hourly 2m temperature")

## 1. Forward transform

`temporal.fft` detects the time dimension (`valid_time`) automatically and returns a complex-valued result indexed by a `frequency` dimension.

Because `valid_time` is a datetime coordinate, the sample spacing is known in seconds and the frequency coordinate is therefore in Hz. That is worth stating explicitly: had the time axis been numeric — *hours since* some epoch, say — the frequencies would be in the reciprocal of *those* units, and `temporal.fft` would leave the coordinate unlabelled rather than assert a unit it cannot verify.

The forward transform also records which dimension it consumed, so the inverse transforms further down restore `valid_time` without being told its name.

In [ ]:
spectrum = ekt.temporal.fft(da)

spectrum

## 2. Power spectrum and the dominant cycle

The squared amplitude of each frequency component gives the power spectrum. We keep the positive frequencies and read the period of each component directly from the `period` coordinate that `temporal.fft` attaches (`1 / frequency`). Because the frequencies are in Hz, the period is a `timedelta64` duration, which we express in hours to make the result easier to interpret.

In [ ]:
positive = spectrum.where(spectrum["frequency"] > 0, drop=True)
power = (np.abs(positive) ** 2).rename("power")

# The `period` coordinate is a timedelta; express it in hours for readability
period_hours = positive["period"] / np.timedelta64(1, "h")
power = power.assign_coords(period_hours=("frequency", period_hours.data))

# Dominant period for each timeseries (one per location)
dominant_period = period_hours.isel(frequency=power.argmax("frequency"))
for location in dominant_period["location"].values:
    period = float(dominant_period.sel(location=location))
    print(f"{location}: dominant period {period:.1f} hours")

power.plot.line(x="period_hours", hue="location", marker="o")
plt.xlabel("Period (hours)")
plt.title("Power spectrum")

The strongest peak is close to 24 hours, reflecting the diurnal cycle of near-surface temperature.

## 3. Centre the spectrum with `fftshift`

The raw FFT output orders the frequencies from zero upwards and then wraps around to the negative frequencies. `temporal.fftshift` reorders the data (and its frequency coordinate) so that the zero-frequency component is in the centre, which is often more convenient for plotting. Its inverse, `temporal.ifftshift`, restores the original ordering.

In [ ]:
shifted = ekt.temporal.fftshift(spectrum, freq_dim="frequency")

np.abs(shifted).plot.line(x="frequency", hue="location", marker="o")
plt.title("Amplitude spectrum (zero frequency centred)")

# ifftshift undoes the reordering
restored_order = ekt.temporal.ifftshift(shifted, freq_dim="frequency")
print("fftshift round-trips:", bool(np.allclose(restored_order, spectrum)))

## 4. Low-pass filter and inverse transform

We can filter in the frequency domain and transform back to the time domain. Here we keep only the low-frequency components (periods of 12 hours or longer) by setting the higher frequencies to zero, then reconstruct a smoothed signal with `temporal.ifft`.

In [ ]:
# Keep components with a period of 12 hours or longer
cutoff_hz = 1.0 / (12 * 3600.0)
filtered = spectrum.where(np.abs(spectrum["frequency"]) <= cutoff_hz, 0)

smoothed = ekt.temporal.ifft(filtered, time_coord=da["valid_time"].values).real

# One panel per location so the original and filtered signals stay legible
fig, axes = plt.subplots(1, da.sizes["location"], figsize=(12, 4), sharey=True)
for ax, location in zip(np.atleast_1d(axes), da["location"].values):
    da.sel(location=location).plot.line(x="valid_time", marker="o", label="original", ax=ax)
    smoothed.sel(location=location).plot.line(x="valid_time", label="low-pass filtered", ax=ax)
    ax.set_title(str(location))
    ax.legend()
fig.tight_layout()

## 5. Round-trip verification

Applying the inverse transform to the full spectrum recovers the original signal (up to floating-point precision — the data are `float32`, and the transforms preserve that precision rather than promoting to `float64`). Note that `time_dim` is not needed: `temporal.ifft` restores the `valid_time` dimension that `temporal.fft` recorded.

In [ ]:
restored = ekt.temporal.ifft(spectrum, time_coord=da["valid_time"].values).real

print("Maximum absolute difference:", float(np.abs(restored - da).max()))

## 6. Real-valued transforms with `rfft`/`irfft`

The temperature signal is real-valued, so we can use `temporal.rfft` instead of `temporal.fft`. It returns only the non-negative frequencies (length `n // 2 + 1`), which roughly halves the size of the output by dropping the redundant negative-frequency half. The inverse `temporal.irfft` reconstructs the real signal directly, without needing to take the real part.

A half-spectrum of length `m` could have come from a signal of `2m - 2` or `2m - 1` points, so inverting one is ambiguous in general. `temporal.rfft` records the length it started from, so `temporal.irfft` recovers it exactly — here 72 points, but equally 71 had the series been odd. Pass `n` when inverting a spectrum that this module did not produce, or one that has been sliced.

In [ ]:
rspectrum = ekt.temporal.rfft(da)

print(f"fft frequencies:  {spectrum.sizes['frequency']}")
print(f"rfft frequencies: {rspectrum.sizes['frequency']}")

rpower = (np.abs(rspectrum) ** 2).rename("power")
rperiod_hours = rspectrum["period"] / np.timedelta64(1, "h")
rpower = rpower.assign_coords(period_hours=("frequency", rperiod_hours.data))

rpower.where(rspectrum["frequency"] > 0, drop=True).plot.line(x="period_hours", hue="location", marker="o")
plt.xlabel("Period (hours)")
plt.title("Power spectrum (rfft)")

In [ ]:
restored_r = ekt.temporal.irfft(rspectrum, time_coord=da["valid_time"].values)

print("Maximum absolute difference:", float(np.abs(restored_r - da).max()))

## 7. Building frequency axes with `fftfreq`/`rfftfreq`

The sample-frequency helpers return the frequency coordinate for a given window length and sample spacing as a standalone `xarray.DataArray`, without computing a transform. They also carry the matching `period` coordinate (`1 / frequency`). This is handy for labelling or selecting frequencies. `fftfreq` matches the output of `fft` (positive and negative frequencies) while `rfftfreq` matches `rfft` (non-negative only). With hourly data the sample spacing is 3600 seconds.

In [ ]:
n = da.sizes["valid_time"]
freqs = ekt.temporal.rfftfreq(n, sample_spacing=3600.0)

# The helper reproduces the frequency coordinate computed by rfft
print("Matches rfft coordinate:", bool(np.allclose(freqs, rspectrum["frequency"])))

# The `period` coordinate is in seconds; express it in hours
periods = freqs["period"] / 3600.0
periods.plot.line(marker="o")
plt.ylabel("Period (hours)")
plt.title("rfftfreq sample periods")

## 8. Hermitian-symmetric transforms with `ihfft`/`hfft`

Because a real signal has a Hermitian-symmetric spectrum, `temporal.ihfft` returns the compact non-negative-frequency half, and `temporal.hfft` transforms such a Hermitian half back to a real time series. The two are inverses of one another, so applying them in sequence recovers the original signal.

In [ ]:
hermitian = ekt.temporal.ihfft(da)
restored_h = ekt.temporal.hfft(hermitian, time_coord=da["valid_time"].values)

print("Maximum absolute difference:", float(np.abs(restored_h - da).max()))

## Beyond the time dimension

Every transform shown here is also available generically in the `earthkit.transforms.fourier` module, where the dimension is specified explicitly (for example `earthkit.transforms.fourier.fft(dataarray, dim="longitude")`). The n-dimensional transforms (`fftn`/`ifftn`/`rfftn`/`irfftn`) live only in that generic module, and are most useful when transforming over several dimensions at once, for example latitude and longitude together for a 2-D spatial spectrum.